# പാഠം 13 - ഏജന്റ് മമ്മറി


## ക്രമീകരണം

ഈ നോട്ട്‌ബുക്ക് persistent memory ഉപയോഗിച്ച് **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് ഒരു യാത്ര ബുക്കിംഗ്ഗ് ഏജൻറ് എങ്ങനെ നിർമ്മിക്കാമെന്ന് കാണിക്കുന്നു.

കഥാമേഖലകളിൽ ഏജൻറ് വിവരങ്ങൾ എങ്ങനെ պահպանിക്കുകയും ഉപയോഗിക്കുകയും ചെയ്യുന്നതിൽ വേർതിരിവുള്ള ഏജൻറ് മെമ്മറികളുടെ വിവിധ തരം — പ്രവർത്തന, ഷോർട്ട്-ടെർമി, ലോങ്-ടെർമി — എങ്ങനെ ബാധിക്കുന്നുവെന്ന് നിങ്ങൾക്ക് പഠിക്കാം.

**ആവശ്യമായ കാര്യങ്ങൾ:**
- ചാറ്റ് മോഡൽ ഡിപ്ലോയ് ചെയ്‌ത ഒരു Azure AI Foundry പ്രോജക്ട് (ഉദാ. `gpt-4o-mini`).
- Azure CLI ൽ ലോഗിൻ ചെയ്തിരിക്കണം — നിങ്ങളുടെ ടെർമിനലിൽ `az login` എന്നിവ പ്രവർത്തിപ്പിക്കുക.
- `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Azure AI Foundry പ്രോജക്ടിന്റെ എൻഡ്പോയിന്റ്.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ഡിപ്ലോയ് ചെയ്ത മോഡലിന്റെ പേര്.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ഏജന്റ് ഓർമ്മയുടെ തരം

AI ഏജന്റുകൾ വ്യത്യസ്ത തരത്തിലുള്ള ഓർമ്മകൾ ഉപയോഗിക്കാൻ കഴിയും, ഓരോതും വ്യത്യസ്തമായ ഒരു ലക്ഷ്യം സേവിക്കുന്നു:

### വർകിംഗ് മെമ്മറി
സംസാര ത്രെഡ് തന്നെ — ഒരു സെഷനിൽ കൈമാറുന്ന സന്ദേശങ്ങൾ. ഏജന്റ് ഒരേ ത്രെഡിൽ മുമ്പത്തെ സന്ദേശങ്ങളെ തിരിച്ചെറ്റി കാണിച്ച് സുസംവാദം നിലനിർത്താനാകും. MAF ൽ ഇത് **`agent.create_session()`** വഴിയാണ് സൃഷ്ടിക്കുന്നത്, ഇത് ഒരു `AgentSession` നൽകുന്നു.

### ഷോർട്ട്-ടെർമ്മ്മറി
ഒരു ടാസ്‌ക്കിനോ സെഷനിനോ വേണ്ടി തുക്കിയ സൂക്ഷിക്കുന്ന വിവരങ്ങൾ, എന്നാൽ സ്ഥിരമായി সংരക്ഷിക്കപ്പെടുന്നവയല്ല. ഉദാഹരണത്തിന്, ഏജന്റ് ഒരു മൾട്ടി-ടേൺ പ്ലാനിംഗ് സംഭാഷണത്തിൽ സത്യങ്ങൾ ശേഖരിച്ചു അവ ഉപയോഗിച്ച് ഫൈനൽ യാത്രാപരിപാടി സൃഷ്ടിക്കാൻ കഴിയും.

### ലോং-ടെർമ്മ്മറി
**സെഷനുകൾക്കു മുകളിലായി** സ്ഥിരമായി നിലനിൽക്കുന്ന അഭിരുചികൾക്കും സത്യങ്ങൾക്കും. മടങ്ങിയ വരുന്ന ഉപഭോക്താവിന് അവരുടെ ഭക്ഷണപരിധികളും യാത്രാ ശൈലിയുമടങ്ങിച്ചെരിപ്പേണ്ടതില്ല. ലോംഗ്-ടെർമ്മ്മറി സാധാരണയായി ഒരു ബാഹ്യ സ്റ്റോറിനാൽ — ഡാറ്റാബേസ്, ഫയൽ അല്ലെങ്കിൽ വെക്ടർ ഇൻഡക്സ് — പിന്തുണച്ചുകൊണ്ട് ഏജന്റിന് ടൂൾസിലൂടെ കാണിക്കുന്നു.


## സെഷനുകളുമായി പ്രവർത്തിക്കുന്ന മെമ്മറി

അളവിലെ ഏറ്റവും എളുപ്പമായ ഓർമ്മ സംസാര സെഷനാണ്. നിങ്ങൾ ഒരേ സെഷൻ (`agent.create_session()` വഴി സൃഷ്ടിച്ചത) അനന്തര `agent.run()` വിളികളിലേക്ക് പ്രദാനം ചെയ്തപ്പോഴ്, ഏജന്റിന് ആ സംഭാഷണത്തിന്റെ മുഴുവൻ ചരിത്രം ദൃശ്യമാണ്, മുൻപ് പറയപ്പെട്ട വിവരങ്ങൾ ഓർക്കാൻ കഴിയും.

ഒരു യാത്രാ ഏജന്റ് സൃഷ്ടിച്ച് പ്രവർത്തിക്കുന്ന മെമ്മറി ഉദാഹരണമായി കാണിക്കാം.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

അജന്റ് ബജറ്റ് ശരിയായി ഓർമ്മിച്ചു കാരണം ഇരുവരുടേയും സന്ദേശങ്ങളും ഒരേ സെഷനിൽ പങ്കുവെക്കുന്നു. ഇത് **കാര്യസ്മൃതി** ആണ് — ഇത് സെഷന്റെ ആയുസിനിടയ്ക്ക് മാത്രമേ നിലനിൽക്കൂ.

### പുതിയ ത്രെഡിൽ എന്ത് സംഭവിക്കുന്നു?

നാം **പുതിയ** സെഷൻ രൂപീകരിച്ചാൽ, അജന്റിന് മുമ്പത്തെ സംഭാഷണത്തിന്റെ ഓർമ്മയില്ല:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ദീർഘകാല ഓർമ്മ പാറ്റേൺ

ഉപയോക്തൃ മുൻഗണനകൾ **സെഷനുകൾക്ക്​ മധ്യമിൽ** ഓർക്കാൻ, സംഭാഷണ ത്രെഡിനോട് പുറമെ നിലനിൽക്കുന്ന സ്ഥിരതയുള്ളൊരു സംഭരണമേഖല ഉണ്ടായിരിക്കണം. ഏജന്റ് ഈ സംഭരണമേഖലയിൽ പ്രവേശിക്കാൻ **ടൂളുകൾ** — വിവരങ്ങൾ സംരക്ഷിക്കാനും ഓഫ്​ തിരിച്ചുചേർക്കാനും ഉപയോഗിക്കാവുന്ന ഫങ്ഷനുകളോ സംവിധാനങ്ങളോ ആണ്.

താഴെ ഒരു ലളിതമായ ഇൻ- മെമ്മറി മുൻഗണന സംഭരണമേഖല നിർവഹിച്ചിരിക്കുന്നു (ഉത്പാദന സാഹചര്യത്തിൽ ഇത് ഡാറ്റാബോദോ വെക്റ്റർ ഇൻഡക്സ് പോലെ ഒന്നോ ഉപയോഗിച്ചിരുന്നു), ഏജന്റിന് ഉപയോഗം സുലഭമാകാൻ ടൂളുകളായി പുറത്തുവിട്ടിരിക്കുന്നു.

### ശില്‌പരേഖ  
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### സീനാരി 1 — ആദ്യമായി ഉപയോഗിക്കുന്ന пользователь годовщина യാത്ര ബുക്ക് ചെയ്യുന്നു

സാറാ ആദ്യമായി സന്ദർശിക്കുന്നു. ഏജന്റ് അവളുടെ ആഗ്രഹങ്ങൾ ടൂളുകളുടെ സഹായത്തോടെ സംഭരിച്ചു അവ സാഹചര്യമനുസരിച്ച് ഹോട്ടലുകൾ ശുപാർശ ചെയ്യണം.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### പശ്ചാത്തലം 2 — സാറাহ് ആഴ്ചകൾക്കുപിന്നാലെ മടങ്ങിവരുന്നു

സാറാഹ് പുതിയ സെഷൻ പണമാവുന്നതുപോലെ ഒരു **പുതിയത് തോപ്പ്** ആരംഭിക്കുന്നു. പ്രവർത്തന സ്മഷൃതി ശൂന്യമാണ്, പക്ഷേ ദീർഘകാല ഇഷ്ടാനുസൃത സ്റ്റോർ അവരുടെ വിവരങ്ങൾ തുറന്നുവെച്ചിട്ടുണ്ട്. ഏജന്റ് അത് പുന:പ്രാപിച്ച് ശുപാർശകൾ വ്യക്തിഗതമാക്കാൻ ഉപയോഗിക്കണമെന്ന് ഉണ്ട്.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത് മൂന്ന് തരത്തിലുള്ള ഏജന്റ് ഓർമകളും മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കുമായി അവ എങ്ങനെ നടപ്പിലാക്കാമെന്ന് ആണ്:

| ഓർമാ തരം | MAF യന്ത്രം | ആയുസ്സ് |
|---|---|---|
| **വർ킹** | `agent.create_session()` | ഒറ്റ സംവാദം |
| **ഷോർട്ട്-ടേം** | ത്രെഡിനുള്ളിൽ സമാഹരിച്ച_Context | ഒറ്റ ടാസ്‌ക് / സെഷൻ |
| **ലോങ്ങ്-ടേം** | `@tool` ഫംഗ്ഷനുകൾ വഴി ആക്സസ് ചെയ്യുന്ന ബാഹ്യ സ്റ്റോർ | സെഷനുകൾക്കിടയിൽ |

### പ്രധാന സന്ദേശങ്ങൾ
1. **`agent.create_session()`** വർകിംഗ് മെമ്മറി നൽകുന്നു — ഏജന്റ് ഒരു സെഷനിൽ മുഴുവൻ സംവാദചരിത്രവും കാണുന്നു.
2. **പുതിയ സെഷനുകളിൽ കോൺടക്സ്റ്റ് നഷ്ടപ്പെടും** — ലോങ്ങ്-ടേം മെമ്മറി ഇല്ലാതെ ഏജന്റ് മുൻസംവാദങ്ങൾ ഓർക്കാൻ കഴിയില്ല.
3. **`@tool` ഫംഗ്ഷനുകൾ** ഉപരിതലവുമായി ഇടപഴകുന്നു — ഇവ ഏജന്റിന് സ്ഥിരതയുള്ള സ്റ്റോറിൽ നിന്ന് വിവരങ്ങൾ സൂക്ഷിക്കാനും തിരികെ ലഭിക്കാനും സഹായിക്കുന്നു.
4. **വ്യക്തിഗതീകരണം സമയംകൊണ്ട് മെച്ചപ്പെടുന്നു** — കൂടുതൽ ഇഷ്ടപ്പെടലുകൾ സൂക്ഷിക്കുന്നത് ഏജന്റിന്റെ ശിപാർശകളെ മെച്ചപ്പെടുത്തും.

### യാഥാർത്ഥ്യ ലോക പ്രയോഗങ്ങൾ
- **കസ്റ്റമർ സർവീസ്**: കസ്റ്റമർ ചരിത്രവും ഇഷ്ടപ്പെടലുകളും ഓർമ്മിക്കുക
- **പേഴ്സണൽ അസിസ്റ്റന്റുകൾ**: ദിവസങ്ങൾ അല്ലെങ്കിൽ ആഴ്ചകൾ കണ്ട്രോൾ ചെയ്യുക
- **ഹെൽത്ത് കെയർ**: രോഗിയുടെ വിവരങ്ങളും ഇഷ്ടങ്ങളും ട്രാക്ക് ചെയ്യുക
- **ഇ-കൊമേഴ്‌സ്**: ചരിത്രം അടിസ്ഥാനമാക്കി വ്യക്തിഹത്യ ഷോപ്പിംഗ്

### അടുത്ത ചുവടു
- ഇൻ-മെമ്മറി ഡിക്‌ഷണറിയുടെ പകരം ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ സ്റ്റോർ (ഉദാഹരണത്തിന് Azure AI Search) ഉപയോഗിക്കുക
- സമയ-സവേദനശേഷിയുള്ള വിവരങ്ങൾക്ക് മെമ്മറി കാലഹരണ്യമായ്ക്കുക
- പങ്കുവെക്കുന്ന മെമ്മറിയോടെയുള്ള മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കുക
- കോഗ്‌നീ നോട്ട്‌ബുക്കിൽ നോളജ്-ഗ്രാഫ് പിന്തുണയുള്ള മെമ്മറി അന്വേഷിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**വിഷയവസ്തു**:  
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നാം കൃത്യതയ്ക്ക് ശ്രമിച്ചുള്ളെങ്കിലും, സ്വയംകൃതമായ വിവർത്തനങ്ങളിൽ പിഴവുകളും തെറ്റായ വിവരങ്ങളും ഉണ്ടാകാമെന്ന് ദയവായി മനസിലാക്കുക. അതിനാൽ, യഥാർത്ഥ ഭാഷയിൽ ഉള്ള പ്രമാണം പ്രസക്തമായ പ്രാമാണിക ഉറവിടമായി കണക്കാക്കപ്പെടണം. അത്യാവശ്യ വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മാനവ വിവർത്തനം ശിപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിനാൽ ഉണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും അതിൽ നിന്നുള്ള പരിനാമങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
